# Lab Submission Dashboard
*SCTC 1013 — Instructor view*

In [ ]:
import json, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import ipywidgets as widgets
from IPython.display import display, clear_output
plt.style.use('ggplot')

# ── data sources (checked in order) ──────────────────────────────────────────
SOURCES = [
    "/home/jovyan/shared-readwrite/submissions.json",      # shared hub folder (all students)
    os.path.expanduser("~/submissions_local.json"),  # instructor local fallback
]

# To also pull from Google Sheets, set these:
SHEET_ID   = None   # e.g. "1BxiMVs0XRA5nFMdKvBdBZjgmUUqptlbs74OgVE2upms"
CREDS_PATH = "/home/jovyan/shared-readwrite/gs_credentials.json"


## Load submissions

In [ ]:
def load_json_sources(paths):
    records = []
    for path in paths:
        if os.path.exists(path):
            try:
                with open(path) as f:
                    data = json.load(f)
                records.extend(data if isinstance(data, list) else [data])
                print(f"  ✓ {path}  ({len(data)} record(s))")
            except Exception as e:
                print(f"  ✗ {path}: {e}")
        else:
            print(f"  – {path}  (not found)")
    return records

def load_sheets(sheet_id, creds_path):
    try:
        import gspread
        gc = gspread.service_account(filename=creds_path)
        ws = gc.open_by_key(sheet_id).sheet1
        rows = ws.get_all_records()
        print(f"  ✓ Google Sheets  ({len(rows)} record(s))")
        return rows
    except Exception as e:
        print(f"  ✗ Google Sheets: {e}")
        return []

print("Loading submissions from:")
records = load_json_sources(SOURCES)
if SHEET_ID and os.path.exists(CREDS_PATH):
    records += load_sheets(SHEET_ID, CREDS_PATH)

if not records:
    print("\n⚠️  No submissions found yet.")
    print("Check that SOURCES paths are correct or configure SHEET_ID above.")
else:
    df = pd.json_normalize(records).drop_duplicates()
    # keep only the most recent submission per user+lab
    df_latest = (df.sort_values("timestamp")
                   .drop_duplicates(subset=["user","lab"], keep="last")
                   .reset_index(drop=True))
    print(f"\n{len(df)} total submission(s), {len(df_latest)} unique student-lab pair(s)")
    print(f"Labs found: {sorted(df['lab'].unique().tolist())}")


## Dashboard

In [ ]:
# ── widgets ───────────────────────────────────────────────────────────────────
lab_opts  = ["All"] + sorted(df["lab"].unique().tolist())
lab_dd    = widgets.Dropdown(options=lab_opts, description="Lab:", layout=widgets.Layout(width="200px"))
view_dd   = widgets.Dropdown(options=["Latest only","All submissions"], description="Show:", layout=widgets.Layout(width="200px"))
out       = widgets.Output()

def make_dashboard(change=None):
    with out:
        clear_output(wait=True)

        # ── filter ────────────────────────────────────────────────────────────
        src = df_latest if view_dd.value == "Latest only" else df
        filt = src if lab_dd.value == "All" else src[src["lab"] == lab_dd.value]

        if filt.empty:
            print("No submissions for this selection.")
            return

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        fig.suptitle(f"{lab_dd.value}  —  {len(filt)} submission(s)", fontsize=13, y=1.02)

        # ── 1. Score distribution histogram ──────────────────────────────────
        ax = axes[0]
        bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100.1]
        ax.hist(filt["score_pct"].astype(float), bins=bins, color="#5DABB5", edgecolor="white", linewidth=0.8)
        ax.axvline(80, color="tomato", linestyle="--", linewidth=1.2, label="80% threshold")
        ax.set_xlabel("Score (%)")
        ax.set_ylabel("Students")
        ax.set_title("Score distribution")
        ax.set_xlim(0, 100)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
        ax.legend(fontsize=8)

        # ── 2. Per-student score bar chart ────────────────────────────────────
        ax = axes[1]
        by_student = filt.sort_values("score_pct", ascending=True)
        colors = ["#5DABB5" if s >= 80 else "tomato" for s in by_student["score_pct"].astype(float)]
        bars = ax.barh(range(len(by_student)), by_student["score_pct"].astype(float),
                       color=colors, edgecolor="white", height=0.7)
        ax.set_yticks(range(len(by_student)))
        labels = [str(n)[:14] for n in by_student["name"]]
        ax.set_yticklabels(labels, fontsize=8)
        ax.axvline(80, color="tomato", linestyle="--", linewidth=1.0)
        ax.set_xlim(0, 105)
        ax.set_xlabel("Score (%)")
        ax.set_title("Score per student")
        for i, (bar, val) in enumerate(zip(bars, by_student["score_pct"].astype(float))):
            ax.text(val + 1, i, f"{val:.0f}%", va="center", fontsize=7)

        # ── 3. Summary stats ──────────────────────────────────────────────────
        ax = axes[2]
        ax.axis("off")
        scores = filt["score_pct"].astype(float)
        n_pass = (scores >= 80).sum()
        stats = [
            ("Submissions",  len(filt)),
            ("Mean score",   f"{scores.mean():.1f}%"),
            ("Median score", f"{scores.median():.1f}%"),
            ("Highest",      f"{scores.max():.1f}%"),
            ("Lowest",       f"{scores.min():.1f}%"),
            ("≥ 80%  (pass)", f"{n_pass}  /  {len(filt)}"),
            ("< 80%  (review)", f"{len(filt)-n_pass}  /  {len(filt)}"),
        ]
        y = 0.95
        for label, val in stats:
            ax.text(0.05, y, label, transform=ax.transAxes, fontsize=10, color="#555")
            ax.text(0.65, y, str(val), transform=ax.transAxes, fontsize=10,
                    fontweight="bold", color="#222")
            y -= 0.13
        ax.set_title("Summary")

        plt.tight_layout()
        plt.show()

        # ── submission table ──────────────────────────────────────────────────
        print()
        show_cols = [c for c in ["name","user","timestamp","score_pct","correct","total"]
                     if c in filt.columns]
        display(filt[show_cols].sort_values("score_pct", ascending=False).reset_index(drop=True))

        # ── open-ended answers ────────────────────────────────────────────────
        ans_cols = [c for c in filt.columns if c.startswith("answers.") or
                    (c.startswith("open") and not c.endswith("_label"))]
        if ans_cols:
            print(f"\n{'─'*60}")
            print("Open-ended responses (one row per student):")
            for _, row in filt.iterrows():
                print(f"\n── {row.get('name','')}  ({row.get('user','')}) ──")
                for col in ans_cols:
                    val = str(row.get(col,"")).strip()
                    if val and val.lower() not in ("nan","answer",""):
                        q = col.replace("answers.","").replace("open","Q").replace("_"," ")
                        print(f"  {q}:\n    {val[:300]}")

lab_dd.observe(make_dashboard, names="value")
view_dd.observe(make_dashboard, names="value")
display(widgets.HBox([lab_dd, view_dd]))
display(out)
make_dashboard()


## Export

In [ ]:
# Export the current filtered view to CSV
# Change lab= to a specific lab name, or leave as 'all'

def export(lab="all"):
    src = df_latest if lab == "all" else df_latest[df_latest["lab"] == lab]
    fname = f"submissions_{lab}.csv"
    src.to_csv(fname, index=False)
    print(f"Saved {len(src)} rows → {fname}")

# export("Lab04")
# export()
